<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 7 · Validación temporal y modelos autorregresivos desde cero

**Unidad 7**

Pronosticar es predecir el **futuro** a partir del pasado, y eso cambia las reglas del juego que usamos desde la unidad 3. Los datos ya no son independientes (la temperatura de hoy se parece mucho a la de ayer), así que barajarlos para validar deja que el modelo «vea el futuro»; y antes de presumir un modelo hay que compararlo con pronósticos muy sencillos que, con frecuencia, son difíciles de vencer. En este laboratorio vas a construir, con NumPy y desde cero, las piezas básicas de un sistema de pronóstico honesto:

- el esquema de validación con **origen móvil**;
- los pronósticos de referencia **ingenuo** y **estacional ingenuo**, y la métrica **MASE**;
- la **función de autocorrelación** muestral;
- la **matriz de atributos rezagados**, sin fuga del futuro;
- el modelo **autorregresivo AR(p)**, ajustado por mínimos cuadrados, y su pronóstico **recursivo** a $h$ pasos.

Al final las juntamos para pronosticar la temperatura de Hermosillo hasta con dos semanas de anticipación, y medimos cuánto se equivoca una validación cruzada con los datos barajados.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Reglas.** Dentro de tus funciones usa solo numpy: `statsmodels` y `sklearn` se usan aquí únicamente para comparar. Las operaciones deben ser vectorizadas; en cada pregunta se indica qué ciclos están permitidos.

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "rediseno-2027"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u07-lab07-pronosticos"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py', 'p7.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/{RAMA}/unidades/07-series-tiempo/laboratorios/tests/u07-lab07-pronosticos/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, KFold
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.ar_model import AutoReg

import rdp

rdp.estilo.usar()

## Los datos y el problema

Usaremos la **temperatura media diaria** registrada en la estación meteorológica del aeropuerto de Hermosillo, de 1992 a 2024. Los datos vienen de la *Global Historical Climatology Network – Daily* (GHCN-D) de la NOAA (M. J. Menne et al., *Journal of Atmospheric and Oceanic Technology* 29, 2012; dominio público). La serie tiene 12,054 días, de los cuales faltan 82; los rellenamos con **interpolación lineal** entre el día anterior y el siguiente disponibles, para tener una serie sin huecos.

**El problema.** Hoy es el día $T$ y queremos pronosticar la temperatura media de los próximos $h = 1, 2, \dots, 14$ días, $\hat y_{T+1\mid T}, \dots, \hat y_{T+14\mid T}$, usando solo lo observado hasta hoy. Separamos los datos **en el tiempo**: los años 1992–2019 son el *entrenamiento* y 2020–2024 la *prueba*. Nada de lo que hagamos con el entrenamiento puede usar información de la prueba.

In [ ]:
datos = rdp.datos.cargar("temperatura_hermosillo", parse_dates=["fecha"])
serie = datos.set_index("fecha")["tmed"].asfreq("D")
faltantes = int(serie.isna().sum())
serie = serie.interpolate()

fechas = serie.index
y = serie.to_numpy()
T = len(y)
corte = int(np.searchsorted(fechas, pd.Timestamp("2020-01-01")))  # primer día de la prueba

print(f"{T:,} días, de {fechas[0].date()} a {fechas[-1].date()}; {faltantes} rellenados por interpolación")
print(f"entrenamiento: {corte:,} días (1992–2019) · prueba: {T - corte:,} días (2020–2024)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(fechas[:corte], y[:corte], color=rdp.estilo.AZUL, lw=0.4)
ax.plot(fechas[corte:], y[corte:], color=rdp.estilo.ROJO, lw=0.4)
ax.axvline(fechas[corte], color=rdp.estilo.GRIS, ls="--", lw=1)
ax.text(fechas[corte - 120], 44, "entrenamiento", color=rdp.estilo.TINTA, ha="right")
ax.text(fechas[corte + 120], 44, "prueba", color=rdp.estilo.TINTA)
ax.set_ylim(None, 46.5)
ax.set_title("Temperatura media diaria en el aeropuerto de Hermosillo")
ax.set_xlabel("fecha")
ax.set_ylabel("temperatura media (°C)");

Se ve un ciclo anual enorme (de unos 15 °C en enero a unos 32 °C en julio) y, encima, una variación de un día a otro de varios grados. Ese ciclo anual es la **climatología**: la temperatura «normal» de cada fecha. La estimamos con el entrenamiento ajustando por mínimos cuadrados una suma de senos y cosenos con periodo de un año,

$$c(t) = a_0 + \sum_{k=1}^{3}\Bigl[a_k \cos\bigl(2\pi k\,t\bigr) + b_k \sin\bigl(2\pi k\,t\bigr)\Bigr],$$

con $t$ medido en años (2003.5 es el 2 de julio de 2003). Como $c(t)$ es una función conocida de la **fecha**, podemos evaluarla en cualquier día futuro sin hacer trampa. La diferencia $y_t - c(t)$ es la **anomalía**: cuánto más caliente o más fresco estuvo el día respecto a lo normal.

In [ ]:
anio = (fechas.year + (fechas.day_of_year - 1) / 365.25).to_numpy()


def armonicos(t, K=3):
    """Matriz de diseño [1, cos(2πkt), sin(2πkt)] para k = 1..K."""
    w = 2 * np.pi * np.asarray(t)
    return np.column_stack([np.ones(len(w))] + [f(k * w) for k in range(1, K + 1) for f in (np.cos, np.sin)])


coef_clima = np.linalg.lstsq(armonicos(anio[:corte]), y[:corte], rcond=None)[0]
clima = armonicos(anio) @ coef_clima  # climatología de todos los días (ajustada solo con el entrenamiento)
anomalia = y - clima

print(f"desviación estándar en el entrenamiento: temperatura {y[:corte].std():.2f} °C · anomalía {anomalia[:corte].std():.2f} °C")

## 1. Validación con origen móvil

En la unidad 3 validamos barajando los datos (*KFold*). Con una serie de tiempo eso no reproduce lo que pasará en la realidad: el modelo siempre se usará para pronosticar **después** de los datos con que se entrenó. La **validación con origen móvil** (*rolling origin*) simula exactamente eso. Se elige un **origen** $t$, se entrena con lo anterior a $t$ y se evalúa en los siguientes $h$ días; luego el origen avanza un **salto** y se repite:

- con **ventana creciente**, el entrenamiento son todos los datos anteriores, índices $0, 1, \dots, t-1$;
- con **ventana deslizante** de tamaño $v$, solo los últimos $v$: índices $t-v, \dots, t-1$ (o desde 0, si $t < v$). Sirve cuando el pasado lejano ya no se parece al presente.

En ambos casos la prueba son los índices $t, t+1, \dots, t+h-1$. Los orígenes son $t = $ `inicial`, `inicial + salto`, `inicial + 2·salto`, … mientras la prueba quepa completa en la serie, es decir, mientras $t + h \le n$.

**Pregunta 1.** Escribe `particiones_origen_movil(n, inicial, horizonte, salto=1, ventana=None)`, que devuelve una **lista** de tuplas `(ent, pru)` de arreglos de índices enteros, una por origen, en orden. Con `ventana=None` la ventana es creciente; con un entero, deslizante. Si no cabe ningún origen, devuelve la lista vacía. Aquí sí puedes recorrer los orígenes con un ciclo (pero construye cada arreglo de índices con `np.arange`).

In [ ]:
def particiones_origen_movil(n, inicial, horizonte, salto=1, ventana=None):
    """Lista de (índices de entrenamiento, índices de prueba) para validación con origen móvil."""
    ...


for ent, pru in particiones_origen_movil(12, inicial=5, horizonte=3, salto=2):
    print("entrena:", ent, " prueba:", pru)

In [ ]:
grader.check("p1")

Así se ven los primeros orígenes del esquema que usaremos con los datos de prueba (ventana creciente, horizonte de 14 días y salto de 7 días). Para que la figura se lea, dibujamos solo las cuatro semanas anteriores al primer origen.

In [ ]:
particiones = particiones_origen_movil(T, inicial=corte, horizonte=14, salto=7)
fig, ax = plt.subplots(figsize=(10, 3.8))
for i, (ent, pru) in enumerate(particiones[:6]):
    ax.barh(i, len(ent[ent >= corte - 28]), left=max(ent[0], corte - 28) - corte, color=rdp.estilo.AZUL, height=0.6)
    ax.barh(i, len(pru), left=pru[0] - corte, color=rdp.estilo.ROJO, height=0.6)
ax.invert_yaxis()
ax.set_yticks(range(6), [f"origen {i + 1}" for i in range(6)])
ax.set_title("Validación con origen móvil: entrenamiento en azul, prueba en rojo")
ax.set_xlabel("días desde el 1 de enero de 2020 (el entrenamiento sigue hacia atrás hasta 1992)");
print(f"{len(particiones)} orígenes en la prueba")

## 2. Pronósticos de referencia

Antes de construir un modelo hay que saber qué tan bien lo hacen las ideas más sencillas:

- **Ingenuo**: mañana será como hoy, y pasado mañana también: $\hat y_{T+h\mid T} = y_T$ para todo $h$.
- **Estacional ingenuo** con periodo $m$: lo mismo que hace un ciclo, $\hat y_{T+h\mid T} = y_{T+h-m}$ si $h \le m$. Si $h > m$ se repite el último ciclo completo: en general $\hat y_{T+h\mid T} = y_{T + h - m\,(\lfloor (h-1)/m \rfloor + 1)}$. Con datos diarios y $m = 365$, el pronóstico para el 10 de agosto es la temperatura del 10 de agosto del año pasado; con datos mensuales y $m = 12$, el mismo mes del año anterior.

Con índices de Python desde 0 (el último dato es `y[T-1]`), el pronóstico estacional para $h = 1, \dots, m$ son los últimos $m$ datos en orden, y después se repiten.

**Pregunta 2.** Escribe `pronostico_ingenuo(y, h)` y `pronostico_estacional(y, h, m)`, que reciben la historia `y` (arreglo o `Series`, con al menos $m$ datos) y devuelven un arreglo de longitud `h` con los pronósticos para $T+1, \dots, T+h$. Sin ciclos.

In [ ]:
def pronostico_ingenuo(y, h):
    """Pronóstico ingenuo: repite el último valor h veces."""
    ...


def pronostico_estacional(y, h, m):
    """Pronóstico estacional ingenuo con periodo m."""
    ...


print(pronostico_ingenuo(np.array([3.0, 1.0, 4.0]), 3), pronostico_estacional(np.array([1.0, 2.0, 3.0, 4.0, 5.0]), 5, m=2))

In [ ]:
grader.check("p2")

## 3. Una métrica sin unidades: MASE

El error absoluto medio (MAE) está en grados, y la raíz del error cuadrático medio (RECM) también; el error porcentual (MAPE) parece no tener unidades, pero divide entre $y_t$ y explota cerca de cero (y no es lo mismo en °C que en kelvin). R. Hyndman y A. Koehler (*International Journal of Forecasting* 22, 2006) propusieron el **error absoluto medio escalado** (MASE): dividir el MAE del pronóstico entre el MAE que tuvo, **dentro del entrenamiento**, el pronóstico ingenuo (estacional, con periodo $m$) a un paso:

$$\text{MASE} = \frac{\dfrac{1}{H}\sum_{j=1}^{H} \bigl|y_j - \hat y_j\bigr|}{\dfrac{1}{T-m}\sum_{t=m+1}^{T} \bigl|y_t - y_{t-m}\bigr|},$$

donde el numerador promedia sobre los $H$ pronósticos que se evalúan y el denominador usa solo la serie de entrenamiento $y_1, \dots, y_T$. Un MASE menor que 1 dice que el pronóstico es mejor, en promedio, que el ingenuo a un paso dentro de la muestra. No cambia si se cambian las unidades y está definido mientras la serie de entrenamiento no sea constante.

**Pregunta 3.** Escribe `mase(y_real, y_pred, y_ent, m=1)`, que devuelve el MASE como `float`. `y_real` y `y_pred` tienen la misma forma (puede ser un arreglo de dos dimensiones, por ejemplo orígenes × horizontes: se promedian **todas** las entradas) y `y_ent` es la serie de entrenamiento. Sin ciclos.

In [ ]:
def mase(y_real, y_pred, y_ent, m=1):
    """Error absoluto medio escalado por el MAE del ingenuo (estacional) a un paso en el entrenamiento."""
    ...


mase(np.array([5.0, 7.0]), np.array([4.0, 4.0]), np.array([1.0, 3.0, 2.0, 4.0]))

In [ ]:
grader.check("p3")

Ya podemos comparar tres pronósticos de referencia con validación de origen móvil sobre 2020–2024: el **ingenuo**, el **estacional ingenuo** con $m = 365$ (la temperatura del mismo día del año anterior) y la **climatología** $c(t)$. Para cada origen y cada horizonte $h = 1, \dots, 14$ guardamos el error absoluto, y al final promediamos sobre los orígenes.

In [ ]:
def evaluar(pronosticador, particiones, h=14):
    """Errores absolutos (orígenes × horizontes) de un pronosticador f(índices de entrenamiento, h)."""
    errores = np.empty((len(particiones), h))
    for i, (ent, pru) in enumerate(particiones):
        errores[i] = np.abs(y[pru] - pronosticador(ent, h))
    return errores


metodos = {
    "ingenuo": lambda ent, h: pronostico_ingenuo(y[ent], h),
    "estacional (m = 365)": lambda ent, h: pronostico_estacional(y[ent], h, 365),
    "climatología": lambda ent, h: clima[ent[-1] + 1 : ent[-1] + 1 + h],
}
errores = {nombre: evaluar(f, particiones) for nombre, f in metodos.items()}

print(f"{'método':>22} {'MAE h=1':>8} {'MAE h=7':>8} {'MAE h=14':>9} {'MASE':>6}")
for nombre, e in errores.items():
    print(f"{nombre:>22} {e[:, 0].mean():8.2f} {e[:, 6].mean():8.2f} {e[:, 13].mean():9.2f} {e.mean() / np.mean(np.abs(np.diff(y[:corte]))):6.2f}")

El ingenuo es el mejor a un día, pero se deteriora rápido; la climatología no depende del horizonte y gana después de unos días; el estacional ingenuo es el peor de los tres a cualquier horizonte: el mismo día del año pasado es un pronóstico muy ruidoso. ¿Podemos combinar lo mejor de los dos primeros? Para eso necesitamos medir cuánto «recuerda» la serie su pasado.

## 4. La función de autocorrelación

La **autocorrelación** de rezago $k$ mide qué tanto se parece la serie a sí misma desplazada $k$ pasos. Con $\bar y$ la media de $y_1, \dots, y_T$, la **función de autocorrelación muestral** (ACF) es

$$r_k = \frac{\sum_{t=k+1}^{T} (y_t - \bar y)(y_{t-k} - \bar y)}{\sum_{t=1}^{T} (y_t - \bar y)^2}, \qquad k = 0, 1, \dots, K.$$

Siempre $r_0 = 1$. El denominador es el mismo para todos los $k$ (no se divide entre $T - k$), lo que garantiza que la sucesión $r_k$ se comporte como una autocorrelación de verdad. Si la serie fuera **ruido blanco** (valores independientes), los $r_k$ con $k \ge 1$ serían aproximadamente normales con media 0 y desviación $1/\sqrt{T}$, así que caerían el 95 % de las veces dentro de las bandas $\pm 1.96/\sqrt{T}$.

**Pregunta 4.** Escribe `acf_muestral(y, k_max)`, que devuelve el arreglo $(r_0, r_1, \dots, r_{K})$ de longitud `k_max + 1`. Puedes recorrer los **rezagos** $k$ con un ciclo (son pocos), pero **no** los tiempos $t$: cada $r_k$ es un producto punto entre dos rebanadas de la serie centrada. Debe tardar menos de un par de segundos con 300,000 datos y 60 rezagos. (Cuidado: `np.correlate(z, z, mode="full")` calcula **todos** los rezagos y con series largas es demasiado lento.)

In [ ]:
def acf_muestral(y, k_max):
    """Autocorrelación muestral r_0, ..., r_kmax."""
    ...


acf_muestral(np.array([1.0, 2.0, 3.0, 4.0]), 3)

In [ ]:
grader.check("p4")

Comparemos la ACF de la temperatura con la de las anomalías, en el entrenamiento y hasta dos años de rezago.

In [ ]:
K = 730
r_temp = acf_muestral(y[:corte], K)
r_anom = acf_muestral(anomalia[:corte], K)
banda = 1.96 / np.sqrt(corte)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, r, titulo in [(axs[0], r_temp, "temperatura"), (axs[1], r_anom, "anomalía (temperatura − climatología)")]:
    ax.vlines(range(K + 1), 0, r, color=rdp.estilo.AZUL, lw=0.8)
    ax.axhspan(-banda, banda, color=rdp.estilo.GRIS, alpha=0.3, lw=0)
    ax.axhline(0, color=rdp.estilo.TENUE, lw=0.8)
    ax.set_title(f"ACF de la {titulo}", fontsize=12)
    ax.set_xlabel("rezago k (días)")
axs[0].set_ylabel("autocorrelación $r_k$")
fig.tight_layout()
print("anomalía, r_1..r_5:", r_anom[1:6].round(3))

La ACF de la temperatura es una onda con periodo de un año: refleja el ciclo anual, no la «memoria» de la atmósfera. Al quitar la climatología, la ACF de la anomalía cae rápido: la anomalía de hoy se parece mucho a la de ayer ($r_1 \approx 0.79$), bastante a la de hace una semana, y casi nada a la de hace un mes. Esa memoria corta es justo lo que un modelo autorregresivo aprovecha. (Fíjate en que las barras se quedan un poco por encima de las bandas incluso a rezagos grandes: hay tendencia de largo plazo y variaciones de un año a otro que la climatología fija no captura.)

## 5. Atributos rezagados, sin fuga

Para pronosticar con un modelo de regresión convertimos la serie en una tabla: cada renglón corresponde a un tiempo $t$, el **objetivo** es $y_t$ y los **atributos** son valores anteriores, $y_{t-\ell}$ para un conjunto de rezagos $\ell \in \{\ell_1, \dots, \ell_q\}$, todos $\ge 1$. Por ejemplo, con rezagos $(1, 2, 7)$ el renglón del día $t$ tiene la temperatura de ayer, la de anteayer y la de hace una semana.

El primer tiempo con todos sus rezagos disponibles es $t = L = \max_j \ell_j$ (contando desde 0), así que la tabla tiene $T - L$ renglones. El error típico es desplazar mal un índice y meter $y_t$ (o algo posterior) entre los atributos: el modelo «predice» copiando la respuesta, se ve espectacular en la validación y no sirve para nada. Eso es **fuga del futuro**.

**Pregunta 5.** Escribe `matriz_rezagos(y, rezagos)`, que recibe la serie `y` y una lista de rezagos enteros positivos (en cualquier orden) y devuelve la tupla `(X, objetivo)`: `X` es de $(T - L) \times q$, con la columna $j$ igual a $y_{t - \ell_j}$, y `objetivo` es el vector $(y_L, \dots, y_{T-1})$. Sin ciclos sobre $t$ (un ciclo o una lista por comprensión sobre los rezagos está bien).

In [ ]:
def matriz_rezagos(y, rezagos):
    """Matriz de atributos rezagados X (columna j = y_{t − rezagos[j]}) y vector objetivo y_t."""
    ...


X_demo, obj_demo = matriz_rezagos(np.arange(10.0), [1, 3])
print(np.column_stack([X_demo, obj_demo]))

In [ ]:
grader.check("p5")

## 6. El modelo autorregresivo AR(p)

El modelo **autorregresivo de orden $p$** dice que cada valor es una combinación lineal de los $p$ anteriores más un ruido nuevo:

$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \dots + \phi_p y_{t-p} + \varepsilon_t, \qquad \varepsilon_t \sim (0, \sigma^2) \text{ independientes}.$$

Es una **regresión lineal** cuyas variables son los rezagos $1, \dots, p$ de la propia serie: con la matriz de la pregunta 5 y una columna de unos, los coeficientes $(c, \phi_1, \dots, \phi_p)$ se estiman por **mínimos cuadrados**, igual que en la unidad 4. Este estimador (de mínimos cuadrados condicionales, porque condiciona en los primeros $p$ datos) es el que usa `AutoReg` de `statsmodels`. Si las raíces del polinomio $1 - \phi_1 z - \dots - \phi_p z^p$ están fuera del círculo unitario, el proceso es **estacionario** y su media es $\mu = c / (1 - \phi_1 - \dots - \phi_p)$.

**Pregunta 6.** Escribe `ajustar_ar(y, p)`, que devuelve la tupla `(c, phi)`: la intersección (un `float`) y el arreglo $(\phi_1, \dots, \phi_p)$, donde $\phi_1$ es el coeficiente del rezago 1. Usa tu `matriz_rezagos` y `np.linalg.lstsq`.

In [ ]:
def ajustar_ar(y, p):
    """AR(p) con intersección por mínimos cuadrados. Devuelve (c, phi)."""
    ...


c_ar, phi_ar = ajustar_ar(anomalia[:corte], 2)
print(f"AR(2) de la anomalía: c = {c_ar:.4f}, phi = {phi_ar.round(4)}")

In [ ]:
grader.check("p6")

## 7. Pronosticar $h$ pasos, de manera recursiva

Para el día siguiente basta sustituir los últimos $p$ datos: $\hat y_{T+1\mid T} = c + \phi_1 y_T + \dots + \phi_p y_{T-p+1}$. Para el día $T+2$ necesitaríamos $y_{T+1}$, que no conocemos; la **estrategia recursiva** usa en su lugar el pronóstico $\hat y_{T+1\mid T}$, y así sucesivamente:

$$\hat y_{T+h\mid T} = c + \sum_{j=1}^{p} \phi_j\, \tilde y_{T+h-j}, \qquad \tilde y_s = \begin{cases} y_s & s \le T, \\ \hat y_{s\mid T} & s > T. \end{cases}$$

Como el ruido futuro tiene media cero, esto da el pronóstico de error cuadrático mínimo si el modelo es correcto. Para un AR(1) estacionario se puede resolver la recursión: $\hat y_{T+h\mid T} = \mu + \phi_1^{\,h}\,(y_T - \mu)$, que se acerca a la media $\mu$ conforme crece el horizonte.

**Pregunta 7.** Escribe `pronosticar_ar(y, c, phi, h)`, que devuelve el arreglo $(\hat y_{T+1\mid T}, \dots, \hat y_{T+h\mid T})$ con la estrategia recursiva. `y` es la historia (con al menos $p$ datos). Aquí sí necesitas un ciclo sobre los $h$ pasos; dentro de él, calcula cada pronóstico con un producto punto, no con otro ciclo.

In [ ]:
def pronosticar_ar(y, c, phi, h):
    """Pronóstico recursivo de h pasos de un AR(p) con intersección c y coeficientes phi."""
    ...


pronosticar_ar(anomalia[:corte], c_ar, phi_ar, 5)

In [ ]:
grader.check("p7")

## 8. Juntando las piezas: climatología más AR de la anomalía

Nuestro modelo final pronostica la **anomalía** con un AR(p) y le suma la climatología de los días que se pronostican:

$$\hat y_{T+h\mid T} = c(T+h) + \hat a_{T+h\mid T}.$$

A un día, el AR aprovecha que la anomalía de mañana se parece a la de hoy (como el ingenuo); a muchos días, el pronóstico de la anomalía se va a su media (casi cero) y el modelo se vuelve la climatología. Lo evaluamos con el mismo origen móvil, **reajustando** el AR en cada origen con todos los datos anteriores (es barato: un problema de mínimos cuadrados de unos 10,000 renglones). Elegimos $p$ comparando varios órdenes con el mismo origen móvil, pero **dentro del entrenamiento** (orígenes en 2016–2019): la prueba no se toca hasta tener el modelo decidido.

In [ ]:
def pronosticador_ar(p):
    def f(ent, h):
        c, phi = ajustar_ar(anomalia[ent], p)
        futuro = np.arange(ent[-1] + 1, ent[-1] + 1 + h)
        return clima[futuro] + pronosticar_ar(anomalia[ent], c, phi, h)
    return f


inicio_val = int(np.searchsorted(fechas, pd.Timestamp("2016-01-01")))
part_val = particiones_origen_movil(corte, inicial=inicio_val, horizonte=14, salto=7)
mae_val = {p: evaluar(pronosticador_ar(p), part_val).mean() for p in [1, 2, 3, 5, 10]}
for p, v in mae_val.items():
    print(f"p = {p:>2}: MAE de validación (2016–2019, promedio de h = 1..14) = {v:.4f}")
p_mejor = min(mae_val, key=mae_val.get)
print("orden elegido:", p_mejor)

Las diferencias entre órdenes son de una centésima de grado: casi toda la ganancia viene de los primeros rezagos. Nos quedamos con el de menor error de validación.

In [ ]:
inicio = time.perf_counter()
errores[f"climatología + AR({p_mejor})"] = evaluar(pronosticador_ar(p_mejor), particiones)
print(f"evaluación en la prueba: {time.perf_counter() - inicio:.1f} s")

escala = np.mean(np.abs(np.diff(y[:corte])))
print(f"{'método':>22} {'MAE h=1':>8} {'MAE h=7':>8} {'MAE h=14':>9} {'MASE':>6}")
for nombre, e in errores.items():
    print(f"{nombre:>22} {e[:, 0].mean():8.2f} {e[:, 6].mean():8.2f} {e[:, 13].mean():9.2f} {e.mean() / escala:6.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
horizontes = np.arange(1, 15)
colores = [rdp.estilo.ROJO, rdp.estilo.GRIS, rdp.estilo.DORADO_OSC, rdp.estilo.AZUL]
for (nombre, e), color in zip(errores.items(), colores):
    ax.plot(horizontes, e.mean(axis=0), color=color, marker="o", ms=4)
rdp.estilo.etiquetar(ax, 14.3, {nombre: e[:, 13].mean() for nombre, e in errores.items()})
ax.set_xlim(0.5, 18.5)
ax.set_xticks(horizontes)
ax.set_title("Error de pronóstico contra horizonte (prueba 2020–2024, origen móvil)")
ax.set_xlabel("horizonte h (días)")
ax.set_ylabel("error absoluto medio (°C)");

A un día, el modelo combinado es un poco mejor que el ingenuo; conforme crece el horizonte se va acercando a la climatología, aunque a dos semanas todavía le gana por unas décimas de grado. Es un modelo de unas cuantas líneas que, a todos los horizontes, iguala o mejora al mejor pronóstico de referencia. Nada espectacular, y eso es lo honesto: la atmósfera tiene poca memoria, y los pronósticos meteorológicos de verdad usan modelos físicos y observaciones de toda la región, no solo la historia de una estación.

## 9. ¿Y si hubiéramos validado barajando?

Hagamos el experimento que justifica todo este laboratorio. Queremos pronosticar la temperatura **dentro de una semana**, $y_{t+7}$, con dos modelos de scikit-learn: una regresión lineal y un **árbol de decisión** sin podar (un modelo muy flexible, que puede memorizar). Los atributos son las temperaturas de los últimos siete días, $y_t, \dots, y_{t-6}$, más, en una segunda versión, el **año** $t$ (en años con decimales). Con los datos de entrenamiento estimamos el error de tres maneras:

1. **KFold barajado** con 5 pliegues, como en la unidad 3;
2. **origen móvil** con tu función: cinco orígenes, al principio de 2015, 2016, …, 2019, cada uno con un año de prueba;
3. y lo comparamos con el error **real**: el modelo entrenado con 1992–2019 y evaluado en 2020–2024.

Los atributos se construyen con tu `matriz_rezagos`. Si el objetivo es el día $s = t + 7$, lo que se conoce el día $t$ son $y_{s-7}, \dots, y_{s-13}$: pronosticar a 7 días de manera **directa** es usar solo rezagos de 7 en adelante.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

H = 7
X_sem, obj_sem = matriz_rezagos(y, list(range(H, H + 7)))  # rezagos 7, 8, ..., 13 del día objetivo s
s = np.arange(H + 6, T)                                   # el día objetivo de cada renglón
X_sem_anio = np.column_stack([X_sem, anio[s - H]])        # más el año del día t = s − 7 en que se pronostica
en_ent = s < corte
n_e = int(en_ent.sum())
part_anuales = particiones_origen_movil(n_e, inicial=n_e - 5 * 365, horizonte=365, salto=365)

filas = []
for nombre_x, X_ in [("7 días", X_sem), ("7 días + año", X_sem_anio)]:
    for nombre_m, modelo in [("lineal", LinearRegression()), ("árbol", DecisionTreeRegressor(random_state=0))]:
        Xe, oe = X_[en_ent], obj_sem[en_ent]
        barajado = -cross_val_score(modelo, Xe, oe, cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error").mean()
        movil = -cross_val_score(modelo, Xe, oe, cv=part_anuales, scoring="neg_mean_absolute_error").mean()
        real = np.mean(np.abs(modelo.fit(Xe, oe).predict(X_[~en_ent]) - obj_sem[~en_ent]))
        filas.append((nombre_m, nombre_x, barajado, movil, real))

tabla_fuga = pd.DataFrame(filas, columns=["modelo", "atributos", "KFold barajado", "origen móvil", "real 2020–2024"])
tabla_fuga.round(2)

Fíjate en el renglón del árbol con el año: el KFold barajado promete el error más bajo de los dos árboles, y en el futuro resulta el más alto. Las preguntas de reflexión te piden explicar por qué.

## 10. Reflexión

Estas preguntas no se verifican automáticamente: respóndelas en la celda de texto que sigue a cada una. Si necesitas hacer cuentas, agrega celdas de código.

<!-- BEGIN QUESTION -->

**Reflexión 1.** Con la tabla de la sección 9:

1. Para cada modelo, ¿qué estimación del error se parece más a la del futuro, la del KFold barajado o la del origen móvil? ¿En qué caso es mayor la diferencia?
2. Explica el mecanismo. Cuando se baraja, para un día de validación $s$, ¿qué días quedan en el entrenamiento, y qué tanto se parecen sus objetivos y sus atributos a los del día $s$ (piensa en la ACF de la sección 4)? ¿Por qué un árbol sin podar, y sobre todo uno que puede partir por el año, aprovecha eso, y la regresión lineal casi no?
3. Si hubieras elegido entre «árbol con 7 días» y «árbol con 7 días + año» con KFold barajado, ¿cuál habrías elegido, y qué habría pasado en 2020–2024?
4. C. Bergmeir, R. Hyndman y B. Koo (*Computational Statistics & Data Analysis* 120, 2018) demostraron que la validación cruzada barajada **sí** es válida para modelos puramente autorregresivos cuyos residuos no están correlacionados. ¿Es consistente con lo que ves en la regresión lineal? ¿Por qué eso no basta para justificar barajar con cualquier modelo?

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2.** El pronóstico de un AR a horizonte largo.

1. Para un AR(1) estacionario, $y_t = c + \phi\, y_{t-1} + \varepsilon_t$ con $|\phi| < 1$ y media $\mu = c/(1-\phi)$, demuestra por inducción en $h$ que $\hat y_{T+h\mid T} = \mu + \phi^h (y_T - \mu)$ y que el error de pronóstico es $y_{T+h} - \hat y_{T+h\mid T} = \sum_{j=0}^{h-1} \phi^j \varepsilon_{T+h-j}$. Deduce que su varianza es $\sigma^2 (1 - \phi^{2h})/(1 - \phi^2)$. ¿A qué tiende cuando $h \to \infty$, y por qué eso explica que en la gráfica de la sección 8 la curva del AR se acerque cada vez más a la de la climatología?
2. Con $\phi \approx r_1 \approx 0.79$ (la autocorrelación a un día de la anomalía), ¿a partir de qué horizonte el pronóstico conserva menos del 10 % de la anomalía de hoy?
3. Calcula la anomalía media de **cada año**, de 1992 a 2024. ¿Qué tan distintos son los años entre sí? Relaciónalo con las barras de la ACF de la anomalía que se quedan fuera de las bandas a rezagos de cientos de días, y propón una modificación del modelo que aproveche esa información **sin** usar datos del futuro.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()